In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import mmread, mmwrite

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))
s = samples[0]

In [ ]:
# ambimux_gex.shape, ambimux_atac.shape, ambimux_joint.shape

In [ ]:
ambimux_joint = pd.read_csv(f'{projdir}/csv/demux/ambimux.csv', sep='\t', header=0, index_col=0)
ambimux_joint.shape

In [ ]:
joint_sings = ambimux_joint[ambimux_joint['droplet_type'] == 'singlet']

In [ ]:
### get physical days 
demux_time = pd.read_csv(f'{projdir}/txt/demux_time_map.txt', sep='\t', header=0, index_col=None)
demux_map = dict(zip(demux_time['Sample'], demux_time['TimeSample2']))
joint_sings['sample_assignment'] = [f'{x}_{y}' for x,y in zip(joint_sings['sample'], joint_sings['assignment'])]
joint_sings['time'] = [demux_map[x].split('_')[1] for x in joint_sings['sample_assignment']]
joint_sings['time_num'] = [int(x[1:]) for x in joint_sings['time']]

joint_sings['time_assignment'] = [f'{x}_{y}' for x,y in zip(joint_sings['time'], joint_sings['assignment'])]

In [ ]:
joint_sings['time']

In [ ]:
joint_sings['time'].value_counts().sort_index()

In [ ]:
joint_sings['assignment'].value_counts().sort_index()

In [ ]:
out = f'{projdir}/csv/cell_qc/ambimux/sample_donor.csv'
sample_donor = pd.DataFrame(joint_sings['sample_assignment'].value_counts().sort_index())

sample_donor['sample'] = [x.split('_')[0] for x in sample_donor.index]
sample_donor['donor'] = [x.split('_')[1] for x in sample_donor.index]
sample_donor[['sample', 'donor', 'sample_assignment']].to_csv(out, sep='\t', index=False, header=True)
sample_donor

In [ ]:
out = f'{projdir}/csv/cell_qc/ambimux/day_donor.csv'
time_assign = pd.DataFrame(joint_sings['time_assignment'].value_counts().sort_index())
time_assign['day'] = [x.split('_')[0] for x in time_assign.index]
time_assign['donor'] = [x.split('_')[1] for x in time_assign.index]
time_assign[['day', 'donor', 'time_assignment']].to_csv(out, sep='\t', index=False, header=True)

In [ ]:
np.mean(joint_sings['best_rna_ambient']), \
np.mean(joint_sings['best_atac_ambient']), \
np.median(joint_sings['best_rna_ambient']), \
np.median(joint_sings['best_atac_ambient'])

In [ ]:
rna_amb = 0.30
atac_amb = 0.20

rna_mask = joint_sings['best_rna_ambient'] < rna_amb
atac_mask = joint_sings['best_atac_ambient'] < atac_amb

np.sum(rna_mask), np.sum(atac_mask), np.sum((rna_mask & atac_mask))

### write barcodes out

In [ ]:
barcodes = joint_sings[['barcode']].sort_values(by='barcode')
barcodes.to_csv(f'{projdir}/txt/barcodes/ambimux/wgs/singlets.txt',
                                 sep='\t', header=False, index=False)

In [ ]:
barcodes

In [ ]:
for s in samples:
    print(s)
    tmp_joint = joint_sings[joint_sings['sample'] == s]
    tmp_joint[['barcode']].sort_values(by='barcode').to_csv(f'{projdir}/txt/barcodes/ambimux/wgs/sample/{s}.txt',
                                         sep='\t', header=False, index=False)

In [ ]:
for s in samples:
    print(s)
    tmp_joint = joint_sings[joint_sings['sample'] == s]
    for donor in donors:
        sample_joint = tmp_joint[tmp_joint['assignment'] == donor]
        to_write = sample_joint[['barcode']].sort_values(by='barcode')
        to_write.to_csv(f'{projdir}/txt/barcodes/ambimux/wgs/sample_donor/{s}_{donor}.txt',
                                         sep='\t', header=False, index=False)

In [ ]:
times = joint_sings['time'].unique()
for s in times:
    print(s)
    tmp_joint = joint_sings[joint_sings['time'] == s]
    for donor in donors:
        sample_joint = tmp_joint[tmp_joint['assignment'] == donor]
        to_write = sample_joint[['barcode']].sort_values(by='barcode')
        to_write.to_csv(f'{projdir}/txt/barcodes/ambimux/wgs/time_donor/{s}_{donor}.txt',
                                         sep='\t', header=False, index=False)

In [ ]:
joint_sings.to_csv(f'{projdir}/csv/meta/ambimux_time_meta.csv', sep='\t', header=True, index=True)

In [ ]:
demux_map.keys()

In [ ]:
demux_map.values()

In [ ]:
tmp1 = [x.split('_')[0] for x in demux_map.values()]
tmp2 = [x.split('_')[1] for x in demux_map.values()]

new_tmp = [f'{x}_{y}' for x,y in zip(tmp2, tmp1)]
new_tmp[:5]

In [ ]:
tmp_samples = [x.split('_')[0] for x in list(demux_map.keys())]
tmp_samples[:5]

In [ ]:
# ### get the correct bam file